# Other renderers 03: Kluctl

Kluctl is a deployment tool that wraps Kustomize deployments with targets, Jinja2 templating and a GitOps controller. mxc uses it as one of its engines. A project is a `.kluctl.yml` with targets and a `deployment.yml` tree.


In [ ]:
export HOME=/tmp
mkdir -p /source/work/other-lab/kluctl/web && cd /source/work/other-lab/kluctl
cat > .kluctl.yml <<'YAML'
targets:
  - name: dev
    context: dev
    args: {replicas: 1, env: dev}
  - name: prod
    context: prod
    args: {replicas: 3, env: prod}
args:
  - name: replicas
  - name: env
YAML
cat > deployment.yml <<'YAML'
deployments:
  - path: web
commonLabels:
  app.kubernetes.io/part-of: kluctl-lab
YAML
cat > web/kustomization.yaml <<'YAML'
resources:
  - deployment.yaml
YAML
cat > web/deployment.yaml <<'YAML'
apiVersion: apps/v1
kind: Deployment
metadata:
  name: web
  namespace: web-{{ args.env }}
spec:
  replicas: {{ args.replicas }}
  selector:
    matchLabels: {app: web}
  template:
    metadata:
      labels: {app: web}
    spec:
      containers:
        - name: web
          image: traefik/whoami:v1.11.0
YAML
find . -type f | sort


`kluctl render` evaluates the Jinja2 templates and Kustomize for a target. The targets reference kubeconfig contexts; without a cluster the render still shows what would be applied (or explains what it needs).


In [ ]:
cd /source/work/other-lab/kluctl
export HOME=/tmp
kluctl render -t prod --no-update-check --render-output-dir /tmp/kluctl-prod 2>&1 | tail -4; find /tmp/kluctl-prod -name '*.yml' -o -name '*.yaml' 2>/dev/null | head -5; (grep -rh 'replicas' /tmp/kluctl-prod 2>/dev/null | head -2) || true


Compared with the tools before it: Kluctl adds a deployment lifecycle (diff, prune, delete, GitOps controller) on top of Kustomize plus templating, the same slot Helm fills for charts and Timoni for CUE modules.
